# 03 · The LSTM as a Tool for an LLM

The trained LSTM becomes a **function the LLM can call**. The LLM decides when to run it, reads the structured
result and explains it — optionally grounded in documents retrieved from a **persistent local Chroma** vector store.

1. Wrap the model as a tool and run a minimal tool-calling loop
2. Add RAG: ingest `docs/ecg_reference.md` into Chroma and expose search as a second tool
3. Use the same `Agent` the FastAPI app serves
4. Call the running FastAPI service over HTTP

Requires a filled-in `.env` (see `.env.example`) and the model from `02_lstm_training.ipynb`.
Works with any OpenAI-compatible endpoint (OpenAI, Ollama, vLLM, …) as long as the model supports tool calling.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
DATA_DIR = ROOT / "data" / "mitdb"
MODEL_DIR = ROOT / "models"

In [ ]:
import json
import numpy as np

from openai import OpenAI

from app.config import get_settings
from app.ecg import mitbih
from app.ecg.classifier import ECGClassifier

settings = get_settings()
llm = OpenAI(api_key=settings.llm_api_key, base_url=settings.llm_base_url)
clf = ECGClassifier(settings.resolve(settings.ecg_model_path), settings.resolve(settings.ecg_meta_path))
print("LLM:", settings.llm_model, "| ECG classes:", clf.classes)

## 1. A minimal tool: `analyze_ecg`
The LLM cannot pass 20,000 samples as an argument, so recordings live in a registry and the tool takes an id.

In [ ]:
mitbih.download(DATA_DIR)
recordings = {}
for rec_id, seconds in [("233", 60), ("100", 60)]:
    sig, _, _, fs = mitbih.load_record(DATA_DIR, rec_id)
    recordings[f"mitdb_{rec_id}"] = {"signal": sig[: seconds * fs], "fs": fs}

def analyze_ecg(ecg_id: str) -> dict:
    r = recordings.get(ecg_id)
    if r is None:
        return {"error": f"unknown ecg_id {ecg_id}", "available": list(recordings)}
    return clf.analyze(r["signal"], r["fs"])

# Direct call — this is exactly what the LLM will receive
print(json.dumps({k: v for k, v in analyze_ecg("mitdb_233").items() if k != "abnormal_beats"}, indent=2))

In [ ]:
ECG_TOOL = {
    "type": "function",
    "function": {
        "name": "analyze_ecg",
        "description": "Run the LSTM ECG beat classifier on a stored recording. Returns heart rate, RR variability, AAMI beat counts and abnormal beat times.",
        "parameters": {
            "type": "object",
            "properties": {"ecg_id": {"type": "string", "enum": list(recordings)}},
            "required": ["ecg_id"],
        },
    },
}

def run_with_tools(question: str, tools: list, handlers: dict, system: str, max_steps: int = 5) -> str:
    messages = [{"role": "system", "content": system}, {"role": "user", "content": question}]
    for _ in range(max_steps):
        msg = llm.chat.completions.create(model=settings.llm_model, messages=messages, tools=tools).choices[0].message
        if not msg.tool_calls:
            return msg.content
        messages.append(msg.model_dump(exclude_none=True))
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments or "{}")
            print(f"→ LLM called {tc.function.name}({args})")
            result = handlers[tc.function.name](**args)
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps(result)})
    return "max steps reached"

answer = run_with_tools(
    "Compare recordings mitdb_233 and mitdb_100. Which one shows more ectopic activity and what kind?",
    tools=[ECG_TOOL],
    handlers={"analyze_ecg": analyze_ecg},
    system="You are a cardiology research assistant. Use the analyze_ecg tool; never guess. Results are not a diagnosis.",
)
print(answer)

## 2. Add RAG over the persistent Chroma store
Embeddings default to a local MiniLM model (no API key). Data persists in `chroma_db/`.

In [ ]:
from app.embeddings import Embedder
from app.ingest import chunk_text
from app.vector_store import VectorStore

store = VectorStore(settings, Embedder(settings))
text = (ROOT / "docs" / "ecg_reference.md").read_text()
chunks = chunk_text(text, settings.chunk_size, settings.chunk_overlap)
store.delete_source("ecg_reference.md")
print("Ingested chunks:", store.add_chunks("ecg_reference.md", chunks), "| collection size:", store.count())

for hit in store.search("What does a wide QRS complex indicate?", top_k=3):
    print(f"{hit['score']:.3f}  {hit['text'][:120]!r}")

In [ ]:
SEARCH_TOOL = {
    "type": "function",
    "function": {
        "name": "search_knowledge_base",
        "description": "Semantic search over cardiology reference documents.",
        "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]},
    },
}

answer = run_with_tools(
    "Analyze mitdb_233. Is the amount of ventricular ectopy clinically significant according to the reference documents?",
    tools=[ECG_TOOL, SEARCH_TOOL],
    handlers={"analyze_ecg": analyze_ecg, "search_knowledge_base": lambda query: {"results": store.search(query, top_k=4)}},
    system="You are a cardiology research assistant. Use analyze_ecg for recordings and search_knowledge_base for clinical thresholds; cite sources. Not a diagnosis.",
)
print(answer)

## 3. The production `Agent` (same code the FastAPI `/chat` endpoint runs)

In [ ]:
from app.agent import Agent, ECGRegistry

registry = ECGRegistry()
sig, _, _, fs = mitbih.load_record(DATA_DIR, "119")
rec = registry.add(sig[: 60 * fs].tolist(), fs, name="mitdb_119 first minute")

agent = Agent(settings, store, registry, clf)
out = agent.chat("Summarise this ECG and explain any abnormal beats in plain language.", ecg_id=rec.id)
print("Tools used:", out["tool_calls"])
print("Sources:", out["sources"])
print(out["answer"])

## 4. Through the FastAPI service

Start the server in a terminal from the project root:

```bash
source env/bin/activate
uvicorn app.main:app --reload
```

In [ ]:
import httpx

API = f"http://localhost:{settings.app_port}"
try:
    with httpx.Client(base_url=API, timeout=120) as http:
        print(http.get("/health").json())
        sig, _, _, fs = mitbih.load_record(DATA_DIR, "208")
        ecg_id = http.post("/ecg/upload", json={"signal": sig[: 30 * fs].tolist(), "fs": fs, "name": "mitdb_208"}).json()["ecg_id"]
        print("Direct analysis:", http.post(f"/ecg/{ecg_id}/analyze").json()["beat_counts"])
        r = http.post("/chat", json={"message": "What does this recording show? Is it concerning?", "ecg_id": ecg_id}).json()
        print(r["tool_calls"]); print(r["answer"])
except httpx.ConnectError:
    print(f"API not running at {API} — start it with `uvicorn app.main:app` and re-run this cell.")